In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.ndimage import gaussian_filter1d
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import sys
import os

sys.path.append(os.path.abspath(".."))
from models import StepModel, RampModel

# Task 1.4 PSTH classifier

This classifier simulates datasets from each model, gets their smoothed PSTHs. It extracts the maximum valueof the first derivative of the PSTH, denoted f1, and the max absolute value of the second derivative, denoted f2, returning these as a NumPy array. We then train a logistic classifier based on f1 and f2. The performance is evaluated and a percentage accuracy returned. I think due to inherent sensitivity of the features to noise, the performance is quite variable, ranging within 60-75%.

In [ ]:

def sample_step_params(T):
    m   = np.random.uniform(T/4, 3*T/4)
    r   = np.random.uniform(0.5, 6.0)
    x0  = np.random.uniform(0.0, 0.5)
    return dict(m=m, r=r, x0=x0)

def sample_ramp_params(): # parameters as given in announcement
    beta  = np.random.uniform(0.0, 4.0)
    lnσ   = np.random.uniform(np.log(0.04), np.log(4.0))
    sigma = np.exp(lnσ)
    x0    = np.random.uniform(0.0, 0.5)
    return dict(beta=beta, sigma=sigma, x0=x0)

def simulate_dataset(model_type, n_trials=400, T=100): # return spikes of shape (n_trials, T) for step or ramp
    if model_type == "step":
        pars  = sample_step_params(T)
        model = StepModel(**pars, Rh=50)
        spikes, *_ = model.simulate(Ntrials=n_trials, T=T, get_rate=False)
    elif model_type == "ramp":
        pars  = sample_ramp_params()
        model = RampModel(**pars, Rh=50)
        spikes, *_ = model.simulate(Ntrials=n_trials, T=T, get_rate=False)
    else:
        raise ValueError("model_type must be 'step' or 'ramp'")
    return spikes.astype(float)

# psth functions
def psth(spikes, smooth_sigma=3):
    mean_counts = spikes.mean(0)
    return gaussian_filter1d(mean_counts, smooth_sigma, mode="nearest")

def dataset_features(spikes): # extract two features: f1 = max 1st derivative and f2 = max absolute 2nd derivative
    p  = psth(spikes)
    d1 = np.diff(p, 1)
    d2 = np.diff(d1, 1)
    f1 = d1.max()
    f2 = np.abs(d2).max()
    return np.array([f1, f2])

# training the classifier
def train_classifier(n_train_each=60, T=100, n_trials=400):
    X, y = [], []
    for label, mtype in enumerate(["step", "ramp"]):
        for _ in range(n_train_each):
            spikes = simulate_dataset(mtype, n_trials=n_trials, T=T)
            X.append(dataset_features(spikes))
            y.append(label)                     # 0 = step, 1 = ramp
    X = np.vstack(X)
    y = np.array(y)

    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(C=1.0, solver="lbfgs"))
    clf.fit(X, y)
    return clf

# evaluating the classifier
def evaluate(clf, n_test_each=40, T=100, n_trials=400):
    correct = 0
    total   = 2 * n_test_each
    for label, mtype in enumerate(["step", "ramp"]):
        for _ in range(n_test_each):
            spikes = simulate_dataset(mtype, n_trials=n_trials, T=T)
            feat   = dataset_features(spikes).reshape(1, -1)
            pred   = clf.predict(feat)[0]
            correct += int(pred == label)
    acc = 100 * correct / total
    print(f"Accuracy: {acc:.1f}%  ({correct}/{total})")
    return acc

# test run
if __name__ == "__main__":
    T = 100    # 1 s trial, 10 ms bins
    n_trials   = 400    # no more than 400 trials per dataset as required
    train_each = 60     # datasets per model for training
    test_each  = 40     # datasets per model for testing

    clf = train_classifier(n_train_each=train_each, T=T, n_trials=n_trials)
    evaluate(clf, n_test_each=test_each, T=T, n_trials=n_trials)


# PSTH classifier with error measures

In [ ]:
#!/usr/bin/env python3
# psth_classifier.py
#
# One-shot evaluation of the PSTH-feature classifier:
#  • confusion matrix
#  • decision-boundary plot
#  • ROC curve (optional but useful)

# Simulation helpers

def sample_step_params(T):
    m   = np.random.uniform(T / 4, 3 * T / 4)
    r   = np.random.uniform(0.5, 6.0)
    x0  = np.random.uniform(0.0, 0.5)
    return dict(m=m, r=r, x0=x0)


def sample_ramp_params():
    beta  = np.random.uniform(0.0, 4.0)
    lnσ   = np.random.uniform(np.log(0.04), np.log(4.0))
    sigma = np.exp(lnσ)
    x0    = np.random.uniform(0.0, 0.5)
    return dict(beta=beta, sigma=sigma, x0=x0)


def simulate_dataset(model_type, n_trials=400, T=100):
    if model_type == "step":
        pars  = sample_step_params(T)
        model = StepModel(**pars, Rh=50)
        spikes, *_ = model.simulate(Ntrials=n_trials, T=T, get_rate=False)
    elif model_type == "ramp":
        pars  = sample_ramp_params()
        model = RampModel(**pars, Rh=50)
        spikes, *_ = model.simulate(Ntrials=n_trials, T=T, get_rate=False)
    else:
        raise ValueError("model_type must be 'step' or 'ramp'")
    return spikes.astype(float)



# Feature extraction

def psth(spikes, smooth_sigma=3):
    """Trial-averaged spike counts after Gaussian smoothing."""
    mean_counts = spikes.mean(0)
    return gaussian_filter1d(mean_counts, smooth_sigma, mode="nearest")


def dataset_features(spikes):
    """Two scalar features of a PSTH: max first-derivative, and max |second-derivative|."""
    p  = psth(spikes)
    d1 = np.diff(p, 1)
    d2 = np.diff(d1, 1)
    f1 = d1.max()
    f2 = np.abs(d2).max()
    return np.array([f1, f2])



# Training

def train_classifier(n_train_each=60, T=100, n_trials=400):
    X, y = [], []
    for label, mtype in enumerate(["step", "ramp"]):
        for _ in range(n_train_each):
            spikes = simulate_dataset(mtype, n_trials=n_trials, T=T)
            X.append(dataset_features(spikes))
            y.append(label)
    X = np.vstack(X)
    y = np.array(y)

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, solver="lbfgs"),
    )
    clf.fit(X, y)
    return clf



# Evaluation – returns arrays for plotting

def evaluate(clf, n_test_each=40, T=100, n_trials=400):
    X_test, y_true, y_pred, y_prob = [], [], [], []
    for label, mtype in enumerate(["step", "ramp"]):
        for _ in range(n_test_each):
            spikes = simulate_dataset(mtype, n_trials=n_trials, T=T)
            feat   = dataset_features(spikes).reshape(1, -1)
            prob   = clf.predict_proba(feat)[0, 1]   # P(label==‘ramp’)
            pred   = int(prob >= 0.5)

            X_test.append(feat.ravel())
            y_true.append(label)
            y_pred.append(pred)
            y_prob.append(prob)

    y_true, y_pred, y_prob = map(np.array, (y_true, y_pred, y_prob))
    acc = 100 * (y_true == y_pred).mean()
    print(f"Accuracy: {acc:.1f}%  ({(y_true == y_pred).sum()}/{len(y_true)})")
    return np.vstack(X_test), y_true, y_pred, y_prob


# Plots

def plot_confusion_and_roc(y_true, y_pred, y_prob, show_roc=True):
    fig, ax = plt.subplots(figsize=(4, 4))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        cm,
        display_labels=["step", "ramp"],
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion matrix")

    if show_roc:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.2f}")
        plt.plot([0, 1], [0, 1], "--", linewidth=1)
        plt.xlabel("False-positive rate")
        plt.ylabel("True-positive rate")
        plt.title("ROC curve")
        plt.legend()
        plt.gca().set_aspect("equal")

    plt.show()


def plot_decision_boundary(clf, X, y):
    x_min, x_max = X[:, 0].min() - 0.003, X[:, 0].max() + 0.003
    y_min, y_max = X[:, 1].min() - 0.0005, X[:, 1].max() + 0.0005
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz   = clf.predict_proba(grid)[:, 1].reshape(xx.shape)

    plt.figure(figsize=(5, 4))
    # Background probability shading
    plt.contourf(xx, yy, zz, levels=20, alpha=0.3)
    # 0.5 contour == decision boundary
    plt.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)

    # Test points
    colors = np.array(["tab:blue", "tab:orange"])
    plt.scatter(X[:, 0], X[:, 1], c=colors[y], edgecolor="k", alpha=0.8)
    plt.xlabel("f₁  (max first derivative)")
    plt.ylabel("f₂  (max |second derivative|)")
    plt.title("Decision boundary with test points")
    plt.tight_layout()
    plt.show()


# Script entry point

if __name__ == "__main__":
    T          = 100      # 1 s trial, 10 ms bins
    n_trials   = 400
    train_each = 60
    test_each  = 40

    clf = train_classifier(
        n_train_each=train_each,
        T=T,
        n_trials=n_trials,
    )
    X_test, y_true, y_pred, y_prob = evaluate(
        clf,
        n_test_each=test_each,
        T=T,
        n_trials=n_trials,
    )
    plot_confusion_and_roc(y_true, y_pred, y_prob, show_roc=True)
    plot_decision_boundary(clf, X_test, y_true)


# Task 1.4 Area Under Fano Classifier

This classifier uses the area-under-curve (AUC) of the Fano factor over time. The AUC is generally larger in the step model. Using this, we generate datasets for step and ramp, and then find a Fano AUC threshold which best separates the distributions (just the midpoint between means). Accuracy is not fantastic - around 60% - could try tuning something? Should provide an explanation of why the accuracy isn't so good

In [ ]:
# NOTE: sampling and dataset generation functions defined and used in the PSTH classifier are reused here

# computing the area under Fano
def integrated_fano(spikes, bin_size=10):

    n_trials, T = spikes.shape
    T_trim = T - (T % bin_size)
    binned = spikes[:, :T_trim].reshape(n_trials, -1, bin_size).sum(axis=-1)

    mean = binned.mean(axis=0)
    var  = binned.var(axis=0, ddof=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        fano = np.where(mean > 0, var / mean, 0.0)

    return fano.sum() * bin_size

# training
def train_threshold(n_datasets=500, n_trials=400, T=100, bin_size=10, seed=0):
    """
    Generate n_datasets datasets from each model, compute their AUCs,
    and return a threshold that best separates the two clusters.
    """
    rng = np.random.default_rng(seed)
    auc_step = []
    auc_ramp = []
    for _ in range(n_datasets):
        spikes = simulate_dataset("step", n_trials, T)
        auc_step.append(integrated_fano(spikes, bin_size))
        spikes = simulate_dataset("ramp", n_trials, T)
        auc_ramp.append(integrated_fano(spikes, bin_size))

    auc_step = np.array(auc_step)
    auc_ramp = np.array(auc_ramp)

    # simple midpoint between the two means
    threshold = 0.5 * (auc_step.mean() + auc_ramp.mean())
    return threshold, auc_step, auc_ramp


def classify_dataset(spikes, threshold, bin_size=10):
    auc = integrated_fano(spikes, bin_size)
    return "step" if auc >= threshold else "ramp"


# function to evaluate accuracy
def evaluate(threshold, n_datasets=100, n_trials=400, T=100, bin_size=10):
    correct = 0
    total   = 2 * n_datasets  # one from each model per loop
    for _ in range(n_datasets):
        if classify_dataset(simulate_dataset("step", n_trials, T),  threshold, bin_size) == "step":
            correct += 1
        if classify_dataset(simulate_dataset("ramp", n_trials, T),  threshold, bin_size) == "ramp":
            correct += 1
    return 100 * correct / total


# test
if __name__ == "__main__":
    THRESH, train_step_auc, train_ramp_auc = train_threshold()
    acc = evaluate(THRESH)
    print(f"Decision threshold (AUC): {THRESH:.1f}")
    print(f"Classification accuracy on fresh test-sets: {acc:.1f}%")


# Area under fano classifier (with confusion matrix, ROC, decision boundary)

In [ ]:
# sampling helpers & simulate_dataset come from the PSTH-classifier code
# Feature: area under the Fano-factor curve

def integrated_fano(spikes, bin_size=10):
    """
    Compute the integral of the Fano factor curve for one dataset.
    Returns a single scalar (higher tends to indicate 'step' in this setup).
    """
    n_trials, T = spikes.shape
    T_trim = T - (T % bin_size)
    binned = spikes[:, :T_trim].reshape(n_trials, -1, bin_size).sum(axis=-1)

    mean = binned.mean(axis=0)
    var  = binned.var(axis=0, ddof=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        fano = np.where(mean > 0, var / mean, 0.0)

    return fano.sum() * bin_size



# Training: choose the best scalar threshold

def train_threshold(
    n_datasets=500,
    n_trials=400,
    T=100,
    bin_size=10,
    seed=0,
):
    """
    Simulate datasets from both models and pick a threshold that
    minimises mis-classification (here: midpoint of the two means).
    Returns the threshold and the training AUC arrays for later plotting.
    """
    rng = np.random.default_rng(seed)
    auc_step = []
    auc_ramp = []
    for _ in range(n_datasets):
        spikes = simulate_dataset("step", n_trials, T)
        auc_step.append(integrated_fano(spikes, bin_size))

        spikes = simulate_dataset("ramp", n_trials, T)
        auc_ramp.append(integrated_fano(spikes, bin_size))

    auc_step = np.array(auc_step)
    auc_ramp = np.array(auc_ramp)

    threshold = 0.5 * (auc_step.mean() + auc_ramp.mean())
    return threshold, auc_step, auc_ramp



# Prediction for a single dataset

def classify_dataset(spikes, threshold, bin_size=10):
    auc_val = integrated_fano(spikes, bin_size)
    return 0 if auc_val >= threshold else 1  # 0 = step, 1 = ramp



# Evaluation – returns arrays for downstream plots

def evaluate(
    threshold,
    n_datasets=100,
    n_trials=400,
    T=100,
    bin_size=10,
):
    """
    Generate fresh test datasets, classify them, and collect:
      • auc_values (continuous scores)
      • y_true      (0=step, 1=ramp)
      • y_pred      (thresholded labels)
    """
    auc_values = []
    y_true     = []
    y_pred     = []

    for label, model in enumerate(["step", "ramp"]):
        for _ in range(n_datasets):
            spikes = simulate_dataset(model, n_trials, T)
            auc_val = integrated_fano(spikes, bin_size)
            pred    = 0 if auc_val >= threshold else 1

            auc_values.append(auc_val)
            y_true.append(label)
            y_pred.append(pred)

    auc_values = np.array(auc_values)
    y_true     = np.array(y_true)
    y_pred     = np.array(y_pred)

    acc = 100 * (y_true == y_pred).mean()
    print(f"Accuracy: {acc:.1f}%  ({(y_true == y_pred).sum()}/{len(y_true)})")
    return auc_values, y_true, y_pred



# Plots

def plot_confusion_and_roc(auc_values, y_true, y_pred):
    # Confusion matrix -----------------------------------------------
    fig, ax = plt.subplots(figsize=(4, 4))
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(
        cm,
        display_labels=["step", "ramp"],
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion matrix")

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, -auc_values)  # use raw score
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.2f}")
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("False-positive rate")
    plt.ylabel("True-positive rate")
    plt.title("ROC curve")
    plt.legend()
    plt.gca().set_aspect("equal")
    plt.show()


def plot_threshold_1d(auc_values, y_true, threshold):
    """
    One-dimensional 'decision-boundary' view:
    scatter all test AUCs on the x-axis (tiny jitter on y for visibility)
    and draw a vertical line at the threshold.
    """
    rng = np.random.default_rng(0)
    y_jitter = rng.uniform(-0.02, 0.02, size=len(auc_values))  # tiny vertical jitter
    colors = np.array(["tab:blue", "tab:orange"])

    plt.figure(figsize=(6, 2.5))
    plt.scatter(
        auc_values,
        y_jitter,
        c=colors[y_true],
        edgecolor="k",
        alpha=0.8,
        s=35,
    )
    plt.axvline(threshold, color="k", linewidth=2, label="threshold")
    plt.yticks([])  # hide y-axis
    plt.xlabel("Integrated Fano factor (AUC)")
    plt.title("Decision boundary (1-D)")
    plt.legend()
    plt.tight_layout()
    plt.show()



# Script entry-point

if __name__ == "__main__":
    BIN_SIZE   = 10
    N_TRIALS   = 400
    T          = 100
    TRAINSETS  = 500
    TESTSETS   = 100

    thresh, step_auc_train, ramp_auc_train = train_threshold(
        n_datasets=TRAINSETS,
        n_trials=N_TRIALS,
        T=T,
        bin_size=BIN_SIZE,
    )
    print(f"Decision threshold (AUC): {thresh:.3g}")

    auc_test, y_true, y_pred = evaluate(
        thresh,
        n_datasets=TESTSETS,
        n_trials=N_TRIALS,
        T=T,
        bin_size=BIN_SIZE,
    )

    plot_confusion_and_roc(auc_test, y_true, y_pred)
    plot_threshold_1d(auc_test, y_true, thresh)
